# MBG YouTube Sentiment Analysis & Text Mining

End-to-end **Text Mining + NLP + Machine Learning + Deep Learning** project untuk menganalisis komentar YouTube terkait Makan Bergizi Gratis (MBG).

### Portfolio objective
Notebook ini dirancang untuk menunjukkan workflow yang dapat dipertanggungjawabkan:

**Data quality → duplicate-leakage control → blinded human validation → preprocessing → EDA/text mining → stratified split → class-imbalance-aware modeling → Optuna tuning → evaluation → error analysis → explainability → 3-class benchmark → exportable results**

> Dataset YouTube merupakan sampel berbasis platform. Hasil project menggambarkan data yang dikumpulkan, bukan opini seluruh populasi.


## 1. Data Collection & Labeling Methodology

### Data lineage

**YouTube comments → raw CSV → cleaning/quality checks → labeled dataset → model-ready deduplicated corpus → train/validation/test**

Repository menyimpan beberapa tahap data:
- `data/mbg_comments_raw.csv`: raw collection
- `data/mbg_comments_clean.csv`: hasil cleaning dan quality checks
- `data/mbg_comments_to_label.csv`: sample untuk proses labeling manual
- `data/mbg_comments_labeled.csv`: current portfolio dataset yang digunakan notebook

### Label methodology

Dataset portfolio saat ini menggunakan **AI-assisted semantic labeling** untuk kolom sentiment:
- `positive`
- `negative`
- `neutral`

Repository juga menyediakan script yang menyiapkan **1.500 komentar untuk manual labeling**. Notebook ini menambahkan **blinded human-validation sample** yang tidak menampilkan label AI kepada annotator.

Notebook **tidak akan mengarang hasil human validation**. Sampai file validation diisi oleh human annotator, statusnya akan ditampilkan sebagai **PENDING**. Setelah diisi dan notebook dijalankan ulang, agreement metrics (accuracy, Macro F1, Cohen's Kappa, confusion matrix) akan dihitung otomatis.

> Exact AI provider/prompt tidak diklaim di notebook karena informasi tersebut tidak terdokumentasi secara eksplisit di repository.


In [ ]:
import os
import re
import html
import json
import random
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    cohen_kappa_score,
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TF_AVAILABLE = False
OPTUNA_AVAILABLE = False

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense, Dropout
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
except ImportError:
    tf = None

try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    optuna = None

if TF_AVAILABLE:
    tf.keras.utils.set_random_seed(SEED)

if OPTUNA_AVAILABLE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "mbg_comments_labeled.csv").exists():
        PROJECT_ROOT = candidate
        break

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "mbg_comments_labeled.csv"

print("Project root :", PROJECT_ROOT)
print("Dataset path :", DATA_PATH)
print("TensorFlow   :", TF_AVAILABLE)
print("Optuna       :", OPTUNA_AVAILABLE)


In [ ]:
# Load and validate the portfolio dataset.
required_columns = ["row_id", "comment", "comment_clean", "published_at", "sentiment"]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

YT_comments = pd.read_csv(DATA_PATH)

missing_required = [c for c in required_columns if c not in YT_comments.columns]
if missing_required:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_required}")

YT_comments["published_at"] = pd.to_datetime(
    YT_comments["published_at"],
    errors="coerce",
    utc=True
)

YT_comments["sentiment"] = (
    YT_comments["sentiment"]
    .astype("string")
    .str.strip()
    .str.lower()
)

VALID_LABELS = ["positive", "negative", "neutral"]
invalid_labels = sorted(set(YT_comments["sentiment"].dropna().unique()) - set(VALID_LABELS))
if invalid_labels:
    raise ValueError(f"Unexpected sentiment labels: {invalid_labels}")

print(f"Rows          : {len(YT_comments):,}")
print(f"Columns       : {len(YT_comments.columns)}")
print(f"Missing dates : {YT_comments['published_at'].isna().sum():,}")
print(f"Duplicate row_id: {YT_comments['row_id'].duplicated().sum():,}")
print(f"Duplicate exact comment: {YT_comments['comment'].duplicated().sum():,}")
print("\nSentiment distribution:")
print(YT_comments["sentiment"].value_counts())


## 2. Data Quality & Duplicate Leakage Control

Duplicate text is preserved in the raw dataset for descriptive analysis, but **must not be allowed to leak across model splits**.

Strategy:
1. Normalize raw comment text into a deterministic `dedup_key`.
2. Remove invalid/empty comments.
3. Detect duplicate groups with conflicting sentiment labels.
4. Remove conflicting duplicate groups from modeling rather than arbitrarily choosing a label.
5. Deduplicate by `dedup_key` **before** train/validation/test splitting.
6. Assert that no duplicate `dedup_key` appears in more than one split.

This means the EDA can describe the collected corpus while the ML corpus is protected from repeated-text leakage.


In [ ]:
def normalize_dedup_text(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.I)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

YT_comments["dedup_key"] = YT_comments["comment"].map(normalize_dedup_text)

quality = {
    "raw_rows": int(len(YT_comments)),
    "missing_comment": int(YT_comments["comment"].isna().sum()),
    "empty_comment": int(YT_comments["dedup_key"].eq("").sum()),
    "duplicate_exact_comment": int(YT_comments["comment"].duplicated().sum()),
    "duplicate_normalized_comment": int(YT_comments["dedup_key"].duplicated().sum()),
    "invalid_published_at": int(YT_comments["published_at"].isna().sum()),
}

valid_for_model = YT_comments[
    YT_comments["dedup_key"].ne("")
    & YT_comments["sentiment"].isin(VALID_LABELS)
].copy()

label_counts_per_group = valid_for_model.groupby("dedup_key")["sentiment"].nunique()
conflicting_groups = set(label_counts_per_group[label_counts_per_group > 1].index)
conflicting_row_count = int(
    valid_for_model["dedup_key"].isin(conflicting_groups).sum()
)

if conflicting_groups:
    valid_for_model = valid_for_model[
        ~valid_for_model["dedup_key"].isin(conflicting_groups)
    ].copy()

model_df = valid_for_model.drop_duplicates("dedup_key", keep="first").copy()

quality["conflicting_duplicate_groups"] = int(len(conflicting_groups))
quality["rows_removed_conflicting_groups"] = conflicting_row_count
quality["rows_removed_duplicate_text"] = int(len(valid_for_model) - len(model_df))
quality["model_rows_after_dedup"] = int(len(model_df))

quality_df = pd.DataFrame({
    "metric": list(quality.keys()),
    "value": list(quality.values())
})
display(quality_df)
quality_df.to_csv(RESULTS_DIR / "data_quality_profile.csv", index=False)

print(f"\nModel corpus: {len(model_df):,} unique comments after leakage control.")


## 3. Blinded Human Validation Sample

A strong portfolio project should independently check a subset of AI-assisted labels.

The notebook creates a **300-row balanced validation sample** when possible (up to 100 rows per AI label). The exported file intentionally contains:
- `validation_id`
- `row_id`
- `comment`
- `published_at`
- blank `human_label`
- blank `human_notes`

The AI label is **not exposed** in the annotation file.

After a human fills `human_label`, rerun this notebook to score agreement. No human-validation score is displayed until labels actually exist.


In [ ]:
HUMAN_SAMPLE_PATH = RESULTS_DIR / "human_validation_sample_for_annotation.csv"

if not HUMAN_SAMPLE_PATH.exists():
    parts = []
    for label in VALID_LABELS:
        group = model_df[model_df["sentiment"] == label]
        n = min(100, len(group))
        if n:
            parts.append(group.sample(n=n, random_state=SEED))
    human_sample = pd.concat(parts, ignore_index=True) if parts else model_df.head(0).copy()
    human_sample = human_sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
    human_sample.insert(0, "validation_id", np.arange(1, len(human_sample) + 1))
    human_sample = human_sample[["validation_id", "row_id", "comment", "published_at"]]
    human_sample["human_label"] = ""
    human_sample["human_notes"] = ""
    human_sample.to_csv(HUMAN_SAMPLE_PATH, index=False, encoding="utf-8-sig")

human_validation = pd.read_csv(HUMAN_SAMPLE_PATH)
human_validation["human_label"] = (
    human_validation["human_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

filled = human_validation["human_label"].isin(VALID_LABELS)
print(f"Human-validation file : {HUMAN_SAMPLE_PATH}")
print(f"Rows prepared          : {len(human_validation):,}")
print(f"Human labels filled    : {int(filled.sum()):,}")

if filled.sum() == 0:
    human_validation_status = "PENDING"
    print("Status: PENDING — isi kolom human_label secara blind sebelum menghitung agreement.")
else:
    scored = human_validation.loc[filled, ["row_id", "human_label"]].merge(
        model_df[["row_id", "sentiment"]],
        on="row_id",
        how="inner"
    )
    y_human = scored["human_label"]
    y_ai = scored["sentiment"]
    human_validation_status = "COMPLETED"

    hv_metrics = pd.DataFrame([{
        "rows_scored": len(scored),
        "accuracy": accuracy_score(y_human, y_ai),
        "macro_f1": f1_score(y_human, y_ai, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_human, y_ai, labels=VALID_LABELS),
    }])
    display(hv_metrics)
    hv_metrics.to_csv(RESULTS_DIR / "human_validation_scoring.csv", index=False)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        y_human,
        y_ai,
        labels=VALID_LABELS,
        display_labels=VALID_LABELS,
        ax=ax,
        cmap="Blues",
        colorbar=False
    )
    ax.set_title("Human vs AI-assisted Labels")
    plt.tight_layout()
    plt.show()

print("Human validation status:", human_validation_status)


## 4. Text Preprocessing

Preprocessing is designed for Indonesian YouTube-style text while preserving sentiment-bearing negations.

Steps:
1. HTML/entity normalization
2. URL / mention removal
3. hashtag symbol removal while keeping the word
4. Unicode normalization
5. lowercasing
6. repeated-character normalization
7. slang normalization
8. Indonesian stopword removal
9. negation preservation
10. optional Sastrawi stemming

Stemming remains disabled by default for faster portfolio execution. The preprocessing output is calculated from the current dataset rather than copied from previous runs.


In [ ]:
NEGATION_WORDS = {
    "tidak", "tak", "bukan", "jangan", "belum",
    "gak", "nggak", "ga", "ngga", "tdk", "kurang"
}

STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "untuk", "pada", "dengan",
    "ini", "itu", "ada", "akan", "atau", "juga", "saja", "sudah",
    "sangat", "lebih", "dalam", "karena", "kalau", "kalo", "jadi",
    "buat", "bagi", "oleh", "sebagai", "dapat", "bisa", "mereka",
    "kita", "kami", "saya", "aku", "anda", "dia", "nya", "lah",
    "pun", "kok", "tuh", "mah", "kan", "aja"
} - NEGATION_WORDS

SLANG = {
    "gk": "gak", "ga": "gak", "nggak": "gak", "ngga": "gak",
    "tdk": "tidak", "bkn": "bukan", "jgn": "jangan", "krn": "karena",
    "knp": "kenapa", "knpa": "kenapa", "tp": "tapi", "tpi": "tapi",
    "bgt": "banget", "bgd": "banget", "gt": "gitu", "jg": "juga",
    "skg": "sekarang", "udh": "sudah", "sdh": "sudah", "dah": "sudah",
    "yg": "yang", "dgn": "dengan", "lg": "lagi", "tau": "tahu",
    "smua": "semua", "org": "orang", "dpt": "dapat", "msh": "masih",
    "utk": "untuk", "lbh": "lebih", "dri": "dari", "dr": "dari",
    "kyk": "kayak", "mkn": "makan", "mslh": "masalah", "bnyk": "banyak",
    "mlh": "malah", "nnti": "nanti", "cuma": "hanya", "abis": "habis",
    "bener": "benar", "emg": "memang", "emang": "memang"
}

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    SASTRAWI_AVAILABLE = True
    stemmer = StemmerFactory().create_stemmer()
except Exception:
    SASTRAWI_AVAILABLE = False
    stemmer = None

USE_STEMMING = False

def clean_for_nlp(text):
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.I)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [SLANG.get(tok, tok) for tok in text.split()]
    tokens = [tok for tok in tokens if tok not in STOPWORDS and len(tok) > 1]

    if not tokens:
        return ""

    text = " ".join(tokens)
    if USE_STEMMING and SASTRAWI_AVAILABLE:
        text = stemmer.stem(text)

    return re.sub(r"\s+", " ", text).strip()

YT_comments["text_source"] = (
    YT_comments["comment_clean"]
    .fillna(YT_comments["comment"])
)

YT_comments["text_preprocessed"] = YT_comments["text_source"].map(clean_for_nlp)
model_df["text_preprocessed"] = (
    model_df["comment_clean"]
    .fillna(model_df["comment"])
    .map(clean_for_nlp)
)

raw_empty = int(YT_comments["text_preprocessed"].eq("").sum())
model_empty = int(model_df["text_preprocessed"].eq("").sum())

print("Sastrawi available :", SASTRAWI_AVAILABLE)
print("Stemming enabled   :", USE_STEMMING)
print("Raw empty after preprocessing   :", raw_empty)
print("Model empty after preprocessing :", model_empty)

display(
    YT_comments[["comment", "text_preprocessed"]]
    .sample(min(10, len(YT_comments)), random_state=SEED)
)


## 5. Exploratory Text Mining

EDA uses the full collected dataset to answer descriptive questions:
- sentiment composition
- comment length
- daily discussion volume
- sentiment trend
- top vocabulary

The EDA is intentionally separated from the modeling corpus so duplicate-leakage handling does not distort the descriptive picture of the collected comments.


In [ ]:
YT_comments["comment_length"] = YT_comments["text_preprocessed"].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(YT_comments["comment_length"].dropna().clip(upper=100), bins=30)
ax.set_title("Distribusi Panjang Komentar")
ax.set_xlabel("Jumlah token (maksimum ditampilkan 100)")
ax.set_ylabel("Jumlah komentar")
plt.tight_layout()
plt.show()

sentiment_counts = YT_comments["sentiment"].value_counts().reindex(VALID_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4.5))
sentiment_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribusi Sentiment Dataset")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Jumlah komentar")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

daily_volume = (
    YT_comments.dropna(subset=["published_at"])
    .assign(date=lambda d: d["published_at"].dt.date)
    .groupby("date")
    .size()
)

fig, ax = plt.subplots(figsize=(12, 4.8))
daily_volume.plot(ax=ax)
ax.set_title("Volume Komentar YouTube per Hari")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Jumlah komentar")
plt.tight_layout()
plt.show()

sentiment_trend = (
    YT_comments.dropna(subset=["published_at"])
    .assign(date=lambda d: d["published_at"].dt.date)
    .pivot_table(index="date", columns="sentiment", values="comment", aggfunc="count", fill_value=0)
    .reindex(columns=VALID_LABELS, fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 5))
for label in VALID_LABELS:
    ax.plot(sentiment_trend.index, sentiment_trend[label], label=label)
ax.set_title("Trend Sentiment per Hari")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Jumlah komentar")
ax.legend()
plt.tight_layout()
plt.show()

all_words = " ".join(YT_comments["text_preprocessed"].dropna())
top_words_overall = pd.DataFrame(
    Counter(all_words.split()).most_common(20),
    columns=["word", "count"]
)
display(top_words_overall)


In [ ]:
dominant_label = sentiment_counts.idxmax()
peak_date = daily_volume.idxmax() if len(daily_volume) else None

print("=== EDA DATA-DRIVEN SUMMARY ===")
print(f"Total collected comments : {len(YT_comments):,}")
print(f"Dominant sentiment       : {dominant_label} ({sentiment_counts[dominant_label] / len(YT_comments) * 100:.2f}%)")
if peak_date is not None:
    print(f"Peak discussion date     : {peak_date} ({int(daily_volume.loc[peak_date]):,} comments)")
print(f"Unique modeling comments : {len(model_df):,}")
print(f"Normalized duplicates removed before modeling: {len(valid_for_model) - len(model_df):,}")

if len(top_words_overall):
    print("\nTop five words:")
    for _, row in top_words_overall.head(5).iterrows():
        print(f"- {row['word']}: {int(row['count']):,}")


## 6. Binary Classification Setup: Positive vs Negative

### Why is neutral excluded here?

The binary task answers a narrower modeling question:

**Can a model separate clearly positive comments from clearly negative comments?**

Neutral is therefore excluded from this specific binary classifier, not treated as useless data. Because neutral is retained in the EDA and a separate **3-class classifier** is built later, the exclusion is methodological rather than an attempt to discard the majority class.

The binary dataset remains strongly imbalanced, so **class weighting, Balanced Accuracy, Macro F1, per-class metrics, and PR-AUC** are reported in addition to accuracy.


In [ ]:
binary_df = model_df[
    model_df["sentiment"].isin(["positive", "negative"])
    & model_df["text_preprocessed"].ne("")
].copy()

# Split at the row level after duplicate removal.
train_val_df, test_df = train_test_split(
    binary_df,
    test_size=0.15,
    random_state=SEED,
    stratify=binary_df["sentiment"]
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1764705882,
    random_state=SEED,
    stratify=train_val_df["sentiment"]
)

def assert_no_duplicate_overlap(*dfs):
    key_sets = [set(df["dedup_key"]) for df in dfs]
    for i in range(len(key_sets)):
        for j in range(i + 1, len(key_sets)):
            overlap = key_sets[i] & key_sets[j]
            assert not overlap, (
                f"Duplicate leakage detected between split {i} and {j}: "
                f"{len(overlap)} groups"
            )

assert_no_duplicate_overlap(train_df, val_df, test_df)

X_train = train_df["text_preprocessed"].reset_index(drop=True)
X_val = val_df["text_preprocessed"].reset_index(drop=True)
X_test = test_df["text_preprocessed"].reset_index(drop=True)
y_train = (train_df["sentiment"] == "positive").astype(int).reset_index(drop=True)
y_val = (val_df["sentiment"] == "positive").astype(int).reset_index(drop=True)
y_test = (test_df["sentiment"] == "positive").astype(int).reset_index(drop=True)

print("Duplicate-key leakage assertion: PASSED")
print(f"Train / validation / test rows: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"Train positive share: {y_train.mean() * 100:.2f}%")
print(f"Validation positive share: {y_val.mean() * 100:.2f}%")
print(f"Test positive share: {y_test.mean() * 100:.2f}%")


In [ ]:
# Shared evaluation helpers.
def select_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.10, 0.90, 81)

    rows = []
    for threshold in grid:
        pred = (proba >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "positive_f1": f1_score(y_true, pred, pos_label=1, zero_division=0),
        })

    threshold_df = pd.DataFrame(rows)
    best = threshold_df.loc[threshold_df["macro_f1"].idxmax()]
    return float(best["threshold"]), threshold_df

def binary_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_positive": precision_score(y_true, pred, pos_label=1, zero_division=0),
        "recall_positive": recall_score(y_true, pred, pos_label=1, zero_division=0),
        "f1_positive": f1_score(y_true, pred, pos_label=1, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
    }

classes_binary = np.array([0, 1])
cw_values = compute_class_weight(
    class_weight="balanced",
    classes=classes_binary,
    y=y_train
)
CLASS_WEIGHTS = {int(c): float(w) for c, w in zip(classes_binary, cw_values)}
print("Class weights:", CLASS_WEIGHTS)


## 7. TF-IDF + Logistic Regression Baseline

The classical baseline is trained first because:
- TF-IDF is highly interpretable,
- Logistic Regression supports class weighting directly,
- it provides a stable benchmark before comparing deep learning,
- and its coefficients can be inspected for explainability.

The vectorizer is **fit on training data only**.


In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=30000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

tfidf_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    random_state=SEED
)

tfidf_model.fit(X_train_tfidf, y_train)

tfidf_val_proba = tfidf_model.predict_proba(X_val_tfidf)[:, 1]
tfidf_threshold, tfidf_threshold_df = select_threshold(y_val, tfidf_val_proba)

tfidf_test_proba = tfidf_model.predict_proba(X_test_tfidf)[:, 1]
tfidf_test_metrics = binary_metrics(
    y_test,
    tfidf_test_proba,
    threshold=tfidf_threshold
)
tfidf_val_metrics = binary_metrics(
    y_val,
    tfidf_val_proba,
    threshold=tfidf_threshold
)

print(f"Selected threshold from validation set: {tfidf_threshold:.3f}")
print("\nValidation metrics:")
display(pd.DataFrame([tfidf_val_metrics]))
print("\nTest metrics:")
display(pd.DataFrame([tfidf_test_metrics]))

tfidf_test_pred = (tfidf_test_proba >= tfidf_threshold).astype(int)

print("\nClassification report — TF-IDF:")
display(pd.DataFrame(
    classification_report(
        y_test,
        tfidf_test_pred,
        target_names=["negative", "positive"],
        output_dict=True,
        zero_division=0
    )
).T)


## 8. Deep Learning: BiLSTM Baseline

The sequence model uses a tokenizer fitted **only on training text**. This is another explicit leakage control: validation and test vocabulary information is not used to build the tokenizer.


In [ ]:
if not TF_AVAILABLE:
    raise ImportError("TensorFlow tidak tersedia. Install requirements.txt sebelum menjalankan deep-learning sections.")

MAX_WORDS = 15000
MAX_LEN = 60

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding="post", truncating="post")

VOCAB_SIZE = min(MAX_WORDS, len(tokenizer.word_index) + 1)

def build_bilstm(vocab_size=VOCAB_SIZE, embedding_dim=64, units=32, dropout=0.25, lr=1e-3):
    tf.keras.backend.clear_session()
    model = Sequential([
        Embedding(vocab_size, embedding_dim),
        Bidirectional(LSTM(units, dropout=dropout)),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

bilstm_baseline = build_bilstm()
bilstm_history = bilstm_baseline.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=64,
    class_weight=CLASS_WEIGHTS,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=0
)

bilstm_val_proba = bilstm_baseline.predict(X_val_seq, verbose=0).ravel()
bilstm_threshold, bilstm_threshold_df = select_threshold(y_val, bilstm_val_proba)

bilstm_test_proba = bilstm_baseline.predict(X_test_seq, verbose=0).ravel()

bilstm_val_metrics = binary_metrics(y_val, bilstm_val_proba, threshold=bilstm_threshold)
bilstm_test_metrics = binary_metrics(y_test, bilstm_test_proba, threshold=bilstm_threshold)

print(f"BiLSTM validation threshold: {bilstm_threshold:.3f}")
display(pd.DataFrame([bilstm_test_metrics]))

bilstm_baseline.save(PROJECT_ROOT / "best_baseline_bilstm.keras")

bilstm_test_pred = (bilstm_test_proba >= bilstm_threshold).astype(int)


## 9. Optuna Tuning — Objective = Validation Macro F1

Optuna is configured so that the optimization target is explicitly:

**maximize Macro F1 on the validation set**

The test set remains untouched during hyperparameter search.

The portfolio workflow runs Optuna by default with a small number of trials so that the repository demonstrates a real executed tuning stage without making the notebook unnecessarily heavy. Set `RUN_OPTUNA=0` for a faster local smoke test.


In [ ]:
RUN_OPTUNA = os.getenv("RUN_OPTUNA", "1") == "1"
OPTUNA_TRIALS = int(os.getenv("OPTUNA_TRIALS", "5"))

optuna_ran = False
optuna_completed_trials = 0
bigru_val_proba = None
bigru_test_proba = None

def build_bigru(vocab_size=VOCAB_SIZE, embedding_dim=64, units=64, dropout=0.30, lr=1e-3):
    tf.keras.backend.clear_session()
    model = Sequential([
        Embedding(vocab_size, embedding_dim),
        Bidirectional(GRU(units, dropout=dropout)),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

if RUN_OPTUNA and not OPTUNA_AVAILABLE:
    raise ImportError("RUN_OPTUNA=1 tetapi Optuna tidak tersedia. Install requirements.txt.")

def optuna_objective(trial):
    tf.keras.utils.set_random_seed(SEED + trial.number)

    embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64])
    units = trial.suggest_categorical("units", [32, 64])
    dropout = trial.suggest_float("dropout", 0.20, 0.45)
    lr = trial.suggest_float("lr", 5e-4, 2e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])

    model = build_bigru(
        embedding_dim=embedding_dim,
        units=units,
        dropout=dropout,
        lr=lr
    )

    model.fit(
        X_train_seq,
        y_train,
        validation_data=(X_val_seq, y_val),
        epochs=6,
        batch_size=batch_size,
        class_weight=CLASS_WEIGHTS,
        callbacks=[EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)],
        verbose=0
    )

    val_proba = model.predict(X_val_seq, verbose=0).ravel()
    val_pred = (val_proba >= 0.5).astype(int)

    score = f1_score(
        y_val,
        val_pred,
        average="macro",
        zero_division=0
    )

    return score

if RUN_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(
        optuna_objective,
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=False
    )

    optuna_ran = True
    optuna_completed_trials = len(study.trials)
    best_bigru_params = dict(study.best_params)

    trials_df = study.trials_dataframe()
    trials_df.to_csv(RESULTS_DIR / "optuna_trials.csv", index=False)
else:
    best_bigru_params = {
        "embedding_dim": 64,
        "units": 64,
        "dropout": 0.30,
        "lr": 1e-3,
        "batch_size": 64
    }

print("Optuna available :", OPTUNA_AVAILABLE)
print("Optuna requested :", RUN_OPTUNA)
print("Optuna executed  :", optuna_ran)
print("Trials completed :", optuna_completed_trials)
print("Best parameters  :", best_bigru_params)


In [ ]:
final_bigru = build_bigru(
    embedding_dim=int(best_bigru_params["embedding_dim"]),
    units=int(best_bigru_params["units"]),
    dropout=float(best_bigru_params["dropout"]),
    lr=float(best_bigru_params["lr"])
)

final_bigru_history = final_bigru.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=12,
    batch_size=int(best_bigru_params.get("batch_size", 64)),
    class_weight=CLASS_WEIGHTS,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=0
)

bigru_val_proba = final_bigru.predict(X_val_seq, verbose=0).ravel()
bigru_threshold, bigru_threshold_df = select_threshold(y_val, bigru_val_proba)

bigru_test_proba = final_bigru.predict(X_test_seq, verbose=0).ravel()

bigru_val_metrics = binary_metrics(
    y_val,
    bigru_val_proba,
    threshold=bigru_threshold
)
bigru_test_metrics = binary_metrics(
    y_test,
    bigru_test_proba,
    threshold=bigru_threshold
)

bigru_test_pred = (bigru_test_proba >= bigru_threshold).astype(int)

final_bigru.save(PROJECT_ROOT / "best_model.keras")

print(f"Selected BiGRU threshold: {bigru_threshold:.3f}")
print("\nBiGRU validation metrics:")
display(pd.DataFrame([bigru_val_metrics]))
print("\nBiGRU test metrics:")
display(pd.DataFrame([bigru_test_metrics]))

print("\nClassification report — BiGRU:")
display(pd.DataFrame(
    classification_report(
        y_test,
        bigru_test_pred,
        target_names=["negative", "positive"],
        output_dict=True,
        zero_division=0
    )
).T)


## 10. Precision-Recall Analysis

PR-AUC is especially useful for the positive class because the binary dataset is highly imbalanced.

We report:
- PR-AUC / Average Precision
- Precision-Recall curve
- Positive-class F1
- Macro F1
- Balanced Accuracy

No model is judged by accuracy alone.


In [ ]:
precision_tfidf, recall_tfidf, _ = precision_recall_curve(y_test, tfidf_test_proba)
precision_bigru, recall_bigru, _ = precision_recall_curve(y_test, bigru_test_proba)

pr_auc_tfidf = tfidf_test_metrics["pr_auc"]
pr_auc_bigru = bigru_test_metrics["pr_auc"]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(recall_tfidf, precision_tfidf, label=f"TF-IDF + LR (AP={pr_auc_tfidf:.3f})")
ax.plot(recall_bigru, precision_bigru, label=f"BiGRU (AP={pr_auc_bigru:.3f})")
ax.set_title("Precision-Recall Curves — Binary Test Set")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()
plt.show()

print(f"TF-IDF PR-AUC / Average Precision : {pr_auc_tfidf:.4f}")
print(f"BiGRU PR-AUC / Average Precision   : {pr_auc_bigru:.4f}")


## 11. Model Comparison

Model selection uses **validation Macro F1**. The test set is then used once for final reporting.

This prevents the test set from becoming a tuning set.


In [ ]:
comparison = pd.DataFrame([
    {
        "model": "TF-IDF + Logistic Regression",
        "validation_macro_f1": tfidf_val_metrics["macro_f1"],
        "test_accuracy": tfidf_test_metrics["accuracy"],
        "test_balanced_accuracy": tfidf_test_metrics["balanced_accuracy"],
        "test_macro_f1": tfidf_test_metrics["macro_f1"],
        "test_pr_auc": tfidf_test_metrics["pr_auc"],
    },
    {
        "model": "BiLSTM baseline",
        "validation_macro_f1": bilstm_val_metrics["macro_f1"],
        "test_accuracy": bilstm_test_metrics["accuracy"],
        "test_balanced_accuracy": bilstm_test_metrics["balanced_accuracy"],
        "test_macro_f1": bilstm_test_metrics["macro_f1"],
        "test_pr_auc": bilstm_test_metrics["pr_auc"],
    },
    {
        "model": "BiGRU + Optuna",
        "validation_macro_f1": bigru_val_metrics["macro_f1"],
        "test_accuracy": bigru_test_metrics["accuracy"],
        "test_balanced_accuracy": bigru_test_metrics["balanced_accuracy"],
        "test_macro_f1": bigru_test_metrics["macro_f1"],
        "test_pr_auc": bigru_test_metrics["pr_auc"],
    }
])

display(comparison.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(comparison["model"], comparison["test_macro_f1"])
ax.set_title("Binary Test Macro F1 by Model")
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

selected_model_name = comparison.loc[
    comparison["validation_macro_f1"].idxmax(), "model"
]
print("Model selected for final error analysis:", selected_model_name)


## 12. Error Analysis

Error analysis focuses on:
- false positives (negative comments predicted as positive)
- false negatives (positive comments predicted as negative)
- performance by comment length

The goal is not only to report a score, but to identify where the model fails.


In [ ]:
if selected_model_name == "TF-IDF + Logistic Regression":
    selected_proba = tfidf_test_proba
    selected_threshold = tfidf_threshold
else:
    selected_proba = bigru_test_proba
    selected_threshold = bigru_threshold

selected_pred = (selected_proba >= selected_threshold).astype(int)

error_df = test_df[
    ["row_id", "comment", "published_at", "sentiment"]
].copy()
error_df["true_label"] = (error_df["sentiment"] == "positive").astype(int).values
error_df["predicted_label"] = selected_pred
error_df["positive_probability"] = selected_proba
error_df["error_type"] = np.select(
    [
        (error_df["true_label"] == 0) & (error_df["predicted_label"] == 1),
        (error_df["true_label"] == 1) & (error_df["predicted_label"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct"
)

false_positives = error_df[error_df["error_type"] == "false_positive"].sort_values(
    "positive_probability", ascending=False
)
false_negatives = error_df[error_df["error_type"] == "false_negative"].sort_values(
    "positive_probability", ascending=True
)

print("=== FALSE POSITIVES ===")
display(false_positives[["row_id", "comment", "positive_probability"]].head(10))

print("\n=== FALSE NEGATIVES ===")
display(false_negatives[["row_id", "comment", "positive_probability"]].head(10))

error_df["token_length"] = (
    error_df["comment"]
    .fillna("")
    .map(lambda x: len(clean_for_nlp(x).split()))
)

error_df["length_bin"] = pd.cut(
    error_df["token_length"],
    bins=[-1, 3, 7, 15, 30, np.inf],
    labels=["0-3", "4-7", "8-15", "16-30", "31+"]
)

error_by_length = (
    error_df.assign(is_error=error_df["error_type"].ne("correct"))
    .groupby("length_bin", observed=False)["is_error"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "error_rate"})
    .reset_index()
)

display(error_by_length)
error_df.to_csv(RESULTS_DIR / "error_analysis.csv", index=False)


## 13. Model Explainability

The TF-IDF + Logistic Regression baseline is directly interpretable through its coefficients:
- positive coefficient → pushes the prediction toward positive
- negative coefficient → pushes the prediction toward negative

This gives a transparent vocabulary-level explanation that can be inspected alongside the deep-learning results.


In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coef = tfidf_model.coef_[0]

top_positive_idx = np.argsort(coef)[-20:][::-1]
top_negative_idx = np.argsort(coef)[:20]

explainability_df = pd.concat([
    pd.DataFrame({
        "feature": feature_names[top_positive_idx],
        "coefficient": coef[top_positive_idx],
        "direction": "positive"
    }),
    pd.DataFrame({
        "feature": feature_names[top_negative_idx],
        "coefficient": coef[top_negative_idx],
        "direction": "negative"
    })
], ignore_index=True)

display(explainability_df)

fig, ax = plt.subplots(figsize=(9, 7))
plot_df = explainability_df.copy()
plot_df["label"] = plot_df["feature"] + " (" + plot_df["direction"] + ")"
ax.barh(plot_df["label"], plot_df["coefficient"])
ax.set_title("Top TF-IDF Logistic Regression Coefficients")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()

explainability_df.to_csv(
    RESULTS_DIR / "explainability_top_terms.csv",
    index=False,
    encoding="utf-8-sig"
)


## 14. Three-Class Classification

The binary task excludes neutral only because it is a focused polarity-detection experiment.

To keep neutral from being discarded entirely, this section trains a **3-class TF-IDF + Logistic Regression** classifier using:
- positive
- negative
- neutral

Class weighting is again applied so the large neutral class does not dominate the optimization objective.


In [ ]:
three_df = model_df[
    model_df["sentiment"].isin(VALID_LABELS)
    & model_df["text_preprocessed"].ne("")
].copy()

train3, test3 = train_test_split(
    three_df,
    test_size=0.15,
    random_state=SEED,
    stratify=three_df["sentiment"]
)
train3, val3 = train_test_split(
    train3,
    test_size=0.1764705882,
    random_state=SEED,
    stratify=train3["sentiment"]
)

# Assert no normalized-text overlap across the 3-class splits as well.
assert_no_duplicate_overlap(train3, val3, test3)

vectorizer3 = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=30000
)

X3_train = vectorizer3.fit_transform(train3["text_preprocessed"])
X3_val = vectorizer3.transform(val3["text_preprocessed"])
X3_test = vectorizer3.transform(test3["text_preprocessed"])

three_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    multi_class="auto",
    random_state=SEED
)
three_model.fit(X3_train, train3["sentiment"])

three_val_pred = three_model.predict(X3_val)
three_test_pred = three_model.predict(X3_test)
three_test_proba = three_model.predict_proba(X3_test)
three_classes = three_model.classes_

three_metrics = {
    "accuracy": accuracy_score(test3["sentiment"], three_test_pred),
    "balanced_accuracy": balanced_accuracy_score(test3["sentiment"], three_test_pred),
    "macro_f1": f1_score(test3["sentiment"], three_test_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(test3["sentiment"], three_test_pred, average="weighted", zero_division=0),
    "macro_pr_auc": average_precision_score(
        label_binarize(test3["sentiment"], classes=three_classes),
        three_test_proba,
        average="macro"
    )
}

print("3-class metrics:")
display(pd.DataFrame([three_metrics]))

print("\n3-class classification report:")
display(pd.DataFrame(
    classification_report(
        test3["sentiment"],
        three_test_pred,
        labels=list(three_classes),
        output_dict=True,
        zero_division=0
    )
).T)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test3["sentiment"],
    three_test_pred,
    labels=list(three_classes),
    display_labels=list(three_classes),
    cmap="Blues",
    colorbar=False,
    ax=ax
)
ax.set_title("3-Class Test Confusion Matrix")
plt.tight_layout()
plt.show()


## 15. Training Curves

Training and validation curves are included to make deep-learning behavior visible and to help identify overfitting/underfitting patterns.


In [ ]:
history_df = pd.DataFrame(final_bigru_history.history)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_df["accuracy"], label="train_accuracy")
ax.plot(history_df["val_accuracy"], label="val_accuracy")
ax.set_title("BiGRU Training vs Validation Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_df["loss"], label="train_loss")
ax.plot(history_df["val_loss"], label="val_loss")
ax.set_title("BiGRU Training vs Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.show()


## 16. Final Results & Portfolio Summary


In [ ]:
binary_reports = {
    "TF-IDF + Logistic Regression": tfidf_test_metrics,
    "BiLSTM baseline": bilstm_test_metrics,
    "BiGRU + Optuna": bigru_test_metrics,
}

summary = {
    "dataset": {
        "raw_rows": int(len(YT_comments)),
        "model_rows_after_dedup": int(len(model_df)),
        "binary_rows": int(len(binary_df)),
        "sentiment_counts": {
            label: int(sentiment_counts[label]) for label in VALID_LABELS
        },
    },
    "data_quality": quality,
    "human_validation_status": human_validation_status,
    "optuna": {
        "requested": RUN_OPTUNA,
        "executed": optuna_ran,
        "completed_trials": int(optuna_completed_trials),
        "best_params": best_bigru_params,
    },
    "binary_models": {
        name: {k: float(v) for k, v in metrics.items()}
        for name, metrics in binary_reports.items()
    },
    "selected_model_for_error_analysis": selected_model_name,
    "three_class": {k: float(v) for k, v in three_metrics.items()},
}

model_comparison_export = comparison.copy()
model_comparison_export.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([tfidf_test_metrics]).to_csv(
    RESULTS_DIR / "tfidf_binary_metrics.csv",
    index=False
)
pd.DataFrame([bilstm_test_metrics]).to_csv(
    RESULTS_DIR / "bilstm_binary_metrics.csv",
    index=False
)
pd.DataFrame([bigru_test_metrics]).to_csv(
    RESULTS_DIR / "bigru_binary_metrics.csv",
    index=False
)
pd.DataFrame([three_metrics]).to_csv(
    RESULTS_DIR / "three_class_metrics.csv",
    index=False
)

with open(RESULTS_DIR / "project_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== PORTFOLIO SUMMARY ===")
print(f"Dataset rows                : {len(YT_comments):,}")
print(f"Unique model rows           : {len(model_df):,}")
print(f"Human validation status     : {human_validation_status}")
print(f"Optuna executed             : {optuna_ran}")
print(f"Optuna trials completed     : {optuna_completed_trials}")
print(f"Selected model (validation) : {selected_model_name}")

display(
    comparison[
        [
            "model",
            "validation_macro_f1",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1",
            "test_pr_auc",
        ]
    ].round(4)
)


## 17. Interpretation & Limitations

### What the project demonstrates
- Data quality checks are separated from modeling.
- Duplicate comments are controlled **before** splitting to prevent text leakage.
- Human validation is explicitly incorporated without fabricating human labels.
- Class imbalance is handled with class weighting and evaluated with Macro F1, Balanced Accuracy, and PR-AUC.
- Optuna tuning, when enabled, optimizes validation Macro F1 and never uses the test set.
- Error analysis and model explainability are part of the modeling workflow.
- Neutral is preserved through a dedicated 3-class benchmark.

### Remaining methodological boundary

The current repository still depends on AI-assisted labels. The notebook therefore reports human-validation status as **PENDING** until an annotator fills the blinded sample. This is preferable to inventing a human agreement score.

The dataset is a YouTube sample and may include slang, sarcasm, typo, spam, duplicated phrasing, and context that is difficult to infer from a single comment. Trends describe the collected comments and do not establish causality or population-wide sentiment.


## 18. Reproducibility Checklist

- Fixed random seed: `42`
- Train/validation/test split uses stratification.
- TF-IDF vectorizers are fitted on training data only.
- Neural tokenizer is fitted on training data only.
- Duplicate normalized text is removed before modeling.
- Conflicting duplicate-label groups are excluded from the ML corpus.
- Class weights are derived from the training set.
- Hyperparameter tuning uses validation Macro F1 only.
- Test data is held out for final evaluation.
- PR-AUC is reported for imbalanced binary classification.
- A blinded human-validation template is exported.
- Result tables and JSON summary are exported to `results/`.


## 19. Portfolio Conclusion

This project presents an end-to-end **Text Mining + NLP + Machine Learning + Deep Learning** workflow for YouTube comments related to MBG.

The final pipeline goes beyond model training: it includes data-quality auditing, duplicate-leakage prevention, blinded label validation workflow, imbalance-aware evaluation, Optuna tuning, PR-AUC, error analysis, explainability, and a 3-class benchmark.

The notebook intentionally distinguishes **what is measured from the collected dataset** from **what still requires independent human verification**, making the project more reproducible and defensible as a portfolio artifact.
